# Lab 6.8 &mdash; Challenge &mdash; A Handbook Q&amp;A Bot

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Build the whole pipeline yourself: load, split, embed, store, retrieve, answer, cite
- Choose every parameter you were given in Labs 6.1&ndash;6.7, and defend each one
- Add a confidence band from the retrieval distance, and decide where to put the line
- Ship something that refuses well, because that is the part your users will test first

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a
> `Document`, a Chroma collection, a retriever, a chain), so they are deterministic and do not
> depend on the chat model. Cells marked **Run it for real** put your code in front of the
> sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **Two different models are in play, and only one of them is billed.** The **chat model**
> (`qwen36-35b-a3b-lab`) answers questions and is reached over the gateway. The **embedding
> model** (`all-MiniLM-L6-v2`, 384 dimensions) turns text into vectors and runs on this pod's
> own CPU &mdash; no key, no gateway, no tokens. Keeping them straight is most of Module 6.

> **Less scaffolding here.** Every piece you need appeared in an earlier lab; this
> one asks you to put them in order and pick the numbers.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap, warnings
from typing import Any, Callable

warnings.filterwarnings("ignore")     # sentence-transformers is chatty on first import

WORK = os.path.join("/tmp", "awmas-lab-6-08")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the CHAT model: qwen, through the sandbox gateway -------------------
# Already configured -- nothing to install, no key to register. Read from the
# environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Chat model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- the EMBEDDING model: local, free, nothing to configure --------------
# all-MiniLM-L6-v2, 384 dimensions. It runs on this pod's CPU and has nothing to do with
# the chat model above: no gateway, no key, no tokens billed. The cache is already warm
# in your sandbox, so the first call is a second or two, not a download.
#
# It is reached through onnxruntime rather than torch, and that is a measured choice
# rather than a taste: same model, same vectors, ~170 MB of memory instead of ~840. Your
# whole sandbox has 2.5 GB for every notebook you leave open, and a kernel you have
# forgotten about is still holding its share.
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 behind LangChain's Embeddings interface.

    Two methods is the whole contract -- which is why a store, a splitter and a chain
    never need to know which model is underneath, or what runtime it uses."""

    def __init__(self):
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2
        self._fn = ONNXMiniLM_L6_V2()

    def embed_documents(self, texts: list) -> list:
        return [[float(x) for x in v] for v in self._fn(list(texts))]

    def embed_query(self, text: str) -> list:
        return [float(x) for x in self._fn([text])[0]]


_emb_cache = {}
def get_embeddings():
    """The embedding model, built once per kernel."""
    if "model" not in _emb_cache:
        _emb_cache["model"] = MiniLMEmbeddings()
    return _emb_cache["model"]

print("work dir   :", WORK)
print("chat model :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

## Concept

Everything in this module, in one object:

```
raw text -> split -> embed -> store -> retrieve (k, filter) -> format with sources
         -> prompt (refuse + cite) -> model -> answer
```

You have built every stage. What is left is the part that is actually hard: choosing the
numbers, and deciding what the thing does when it does not know.

In [ ]:
# ------------------------------------------------- a bigger corpus, as raw text
# Five policy documents. Longer and messier than Lab 6.3's ten neat passages -- which is
# why this one has to be split before it can be indexed.

POLICY_DOCS = [
    {"source": "leave-policy.pdf", "category": "leave", "text": """Leave Policy

Annual Leave: full-time employees receive 24 days per year. Apply at least 3 working days in advance through the HR portal. Unused annual leave cannot be carried forward. Employees with less than 6 months tenure receive prorated leave.

Sick Leave: 12 days per year. Notify your manager by 10 AM on the day of absence. For absences exceeding 2 consecutive days a medical certificate is mandatory. Unused sick leave carries forward up to a maximum of 30 days.

Maternity Leave: 26 weeks paid, and it may start up to 8 weeks before the expected date. Applies after 80 days of continuous employment. Paternity Leave: 2 weeks, to be taken within 6 months of the birth."""},

    {"source": "wfh-policy.pdf", "category": "wfh", "text": """Work From Home Policy

Eligibility: employees who have completed the 6-month probation period. New joiners work from the office for their first 6 months, without exception.

Schedule: up to 3 days per week with team lead approval. Core hours are 10 AM to 4 PM IST. Friday is a mandatory in-office day for every team.

Equipment: the company laptop must be used. A VPN connection is mandatory for internal systems. Reimbursement: 1,500 per month for internet, and up to 10,000 once for an ergonomic chair."""},

    {"source": "expense-policy.pdf", "category": "expense", "text": """Expense Policy

Travel: business travel must be pre-approved by your manager. Submit original receipts within 7 working days of completing the travel. Economy class for domestic flights; business class is allowed for international flights over 6 hours.

Meals: 500 per day during client visits within the country, 3,000 per day for international travel. Team dinners up to 1,000 per person with manager approval.

Equipment: laptops are replaced every 3 years. External monitors up to 15,000. Software licences must be requested through the IT helpdesk and must never be bought directly."""},

    {"source": "tech-guide.pdf", "category": "tech", "text": """Technology Guide

Backend: Python with FastAPI, and Java with Spring Boot. New microservices should use Python unless there is a specific reason otherwise. All APIs follow REST conventions.

Frontend: React is the standard for new work. Angular is maintained for the existing dashboard and admin portal. TypeScript is mandatory.

Data: PostgreSQL is the primary relational database. MongoDB where the schema must flex. Redis for caching and sessions. Infrastructure runs on AWS, described in Terraform, deployed by GitHub Actions."""},

    {"source": "office-directory.pdf", "category": "office", "text": """Office Directory

Bangalore is the headquarters: 5th floor, 200+ staff, every department represented. Cafeteria on the 3rd floor. Basement parking on application to admin. Hours are 9 AM to 6 PM, Monday to Friday.

Mumbai: Worli business district, Tower A, 12th floor. 50 staff, mostly sales, client success and marketing.

Hyderabad: 8th floor, 80 staff. The engineering hub for backend and data services, with 24/7 access for on-call engineers.

Pune: Phase 2, Building C, 4th floor. 40 staff in QA, DevOps and SRE."""},
]

print(f"{len(POLICY_DOCS)} documents, "
      f"{sum(len(d['text']) for d in POLICY_DOCS)} characters")

## Part A &mdash; Build the knowledge base

Turn the five documents into `Document` objects, split them, embed them and store them.
Every one of these appeared in Labs 6.3&ndash;6.5.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

def to_documents() -> list:
    """One Document per entry. The text goes in page_content; source and category are
    metadata, and Lab 6.7 will need both."""
    return BLANK


def splitter():
    """These documents are 600-900 characters and each covers several distinct rules.
    Choose a chunk_size and chunk_overlap that keep one rule per chunk."""
    return RecursiveCharacterTextSplitter(chunk_size=BLANK, chunk_overlap=BLANK)


_kb = {}
def knowledge_base():
    """Given -- build once."""
    if "s" not in _kb:
        chunks = splitter().split_documents(to_documents())
        _kb["chunks"] = chunks
        _kb["s"] = Chroma.from_documents(documents=chunks, embedding=get_embeddings(),
                                         collection_name="handbook_challenge")
    return _kb["s"]

In [ ]:
# --- Self-check: Part A
check("five Documents, one per policy file",
      lambda: len(to_documents()) == 5
              and all(isinstance(d, Document) for d in to_documents()))
check("each carries its source and category",
      lambda: all({"source", "category"} <= set(d.metadata) for d in to_documents()))
check("splitting produces more chunks than documents",
      lambda: (knowledge_base(), len(_kb["chunks"]) > 8)[1],
      "if this is 5 your chunk_size is bigger than the documents")
check("no chunk is so big it covers the whole document",
      lambda: (knowledge_base(),
               max(len(c.page_content) for c in _kb["chunks"]) <= 600)[1])
check("the store holds every chunk",
      lambda: knowledge_base()._collection.count() == len(_kb["chunks"]))
check("a specific fact is findable",
      lambda: any("80 days" in d.page_content
                  for d in knowledge_base().similarity_search("maternity eligibility", k=3)))

guard(lambda: print(f"  {len(to_documents())} documents -> {len(_kb['chunks'])} chunks"))

## Part B &mdash; The chain, with citations and a refusal

Same shape as Lab 6.7. Everything here is yours to write.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def top_k() -> int:
    """Five documents, split into a dozen-odd chunks, and some questions span two rules.
    How many chunks should reach the prompt?"""
    return BLANK


def format_with_sources(docs) -> str:
    """Label each chunk with its source, then its text -- as in Lab 6.7."""
    return BLANK


def bot_prompt():
    """A prompt that (a) answers only from the context, (b) refuses when it cannot, and
    (c) cites the source of each fact in a fixed format."""
    return ChatPromptTemplate.from_template(BLANK)


def bot_chain():
    """Given -- your pieces, assembled."""
    rtv = knowledge_base().as_retriever(search_kwargs={"k": top_k()})
    return ({"context": rtv | format_with_sources, "question": RunnablePassthrough()}
            | bot_prompt() | get_llm() | StrOutputParser())

In [ ]:
# --- Self-check: Part B
def _docs():
    return knowledge_base().similarity_search("annual leave", k=2)

check("k is between 2 and 8",
      lambda: 2 <= top_k() <= 8,
      "one chunk cannot answer a two-rule question; ten will bury the answer in noise")
check("the formatter names each chunk's source",
      lambda: all(d.metadata["source"] in format_with_sources(_docs()) for d in _docs()))
check("and keeps the text",
      lambda: all(d.page_content[:30] in format_with_sources(_docs()) for d in _docs()))
check("the prompt takes exactly context and question",
      lambda: set(bot_prompt().input_variables) == {"context", "question"})
check("the prompt restricts the model to the context",
      lambda: "only" in bot_prompt().format(context="c", question="q").lower())
check("the prompt asks for a citation",
      lambda: "cite" in bot_prompt().format(context="c", question="q").lower())
check("the prompt says what to do when the answer is not there",
      lambda: any(w in bot_prompt().format(context="c", question="q").lower()
                  for w in ("do not have", "don't have", "not contain", "cannot answer")))

## Part C &mdash; Confidence, and where you put the line

The retriever always returns `k` chunks. `similarity_search_with_score` gives you the
distance, and that is the only number that knows the difference between a good hit and the
closest of a bad lot.

Look at the printed distances before you pick your threshold. They are properties of this
corpus and this embedding model, and they do not transfer.

In [ ]:
def confident_below() -> float:
    """A Chroma distance below this means the retrieval genuinely matched.

    Run the calibration cell below FIRST and read the two groups of numbers."""
    return BLANK


def answer_with_confidence(question: str) -> dict:
    """Given -- your threshold, applied."""
    hits = knowledge_base().similarity_search_with_score(question, k=top_k())
    best = hits[0][1] if hits else 99.0
    return {"question": question,
            "confident": best < confident_below(),
            "best_distance": round(best, 4),
            "answer": bot_chain().invoke(question) if llm_ready() else "(model not configured)"}

In [ ]:
# --- Calibration: run this BEFORE choosing your threshold (no gateway needed)
ANSWERABLE = ["How many sick days do I get?",
              "Can a new joiner work from home?",
              "What is the meal allowance abroad?",
              "Which database should a new service use?"]
UNANSWERABLE = ["What is the company share price?",
                "What is my notice period?",
                "Who is the CEO?"]

def _calibrate():
    for label, qs in (("answerable  ", ANSWERABLE), ("unanswerable", UNANSWERABLE)):
        for q in qs:
            d = knowledge_base().similarity_search_with_score(q, k=1)[0][1]
            print(f"  {label} [{d:.4f}] {q}")
guard(_calibrate)

In [ ]:
# --- Self-check: Part C
check("your threshold is a number in the range these distances occupy",
      lambda: 0.2 < confident_below() < 2.0)
check("every answerable question clears it",
      lambda: all(answer_with_confidence(q)["best_distance"] < confident_below()
                  for q in ANSWERABLE),
      "too tight -- you are about to refuse questions the handbook can answer")
check("and the out-of-scope ones do not",
      lambda: not any(answer_with_confidence(q)["best_distance"] < confident_below()
                      for q in UNANSWERABLE),
      "too loose -- an off-topic question is being reported as a confident hit")

In [ ]:
# --- Run it for real -------------------------------------------------------
def run_the_bot():
    for q in ANSWERABLE[:2] + UNANSWERABLE[:1]:
        r = answer_with_confidence(q)
        flag = "OK " if r["confident"] else "LOW"
        print(f"[{flag} {r['best_distance']:.3f}] Q: {q}")
        print(f"           A: {r['answer'].strip()[:220]}\n")

if llm_ready():
    guard(run_the_bot)

In [ ]:
score()

## Your turn

1. **The threshold moves.** Re-split the corpus at `chunk_size=200` and re-run the
   calibration cell. The distances change, and your threshold is now wrong. Any number you
   tune against a corpus is a property of that corpus *and* that chunking &mdash; write that
   on the wall next to the number.
2. **Scoped assistants.** Build a leave-only and a tech-only chain from the same store,
   the way Lab 6.7 did. Ask the tech bot a leave question. It answers from tech chunks
   without complaining, which is worth seeing once.
3. **Refuse before you retrieve.** Right now the chain retrieves, spends the tokens, and
   then the model refuses. Use `confident_below()` to short-circuit *before* the model call.
   Measure what you saved on the three unanswerable questions &mdash; and then argue the
   other side, because a threshold that refuses early also refuses silently.
4. **The honest evaluation.** You have seven labelled questions in `ANSWERABLE` and
   `UNANSWERABLE`. Write the loop that scores your bot on both, and report two numbers, not
   one: how often it answered correctly, and how often it refused when it should have. A
   single accuracy number hides which half broke, and Module 7 is about exactly that.